Import packages

In [1]:
from pulp import *

# Mathematical Formulation for Problem 1

### Parameters
- $c_1,c_2,c_3$: cost per barrel of inputs 1–3, where $c_1=17.25,\; c_2=15.75,\; c_3=17.75$.  
- $o_1,o_2,o_3$: octane ratings, where $o_1=100,\; o_2=87,\; o_3=110$.  
- $a_1,a_2,a_3$: available barrels (thousands), where $a_1=150,\; a_2=350,\; a_3=300$.  
- $p_R=21$: price per barrel of Regular.  
- $p_S=25$: price per barrel of Supreme.  
- $d_R=300$: demand for Regular (thousands).  
- $d_S=450$: demand for Supreme (thousands).  
- $\text{octane}_R=90$: minimum octane for Regular.  
- $\text{octane}_S=97$: minimum octane for Supreme.

### Decision variables
- $x_{11},x_{21},x_{31}$: thousands of barrels of inputs 1–3 used for **Regular**.  
- $x_{12},x_{22},x_{32}$: thousands of barrels of inputs 1–3 used for **Supreme**.

### Objective function

\begin{aligned}
\text{Maximize:}\quad & Z \;=\; p_R(x_{11}+x_{21}+x_{31}) \;+\; p_S(x_{12}+x_{22}+x_{32}) \\
&\qquad -\; c_1(x_{11}+x_{12}) \;-\; c_2(x_{21}+x_{22}) \;-\; c_3(x_{31}+x_{32}).
\end{aligned}

(the expanded numeric form:)

\begin{aligned}
\text{Maximize:}\quad & Z \;=\; 21(x_{11}+x_{21}+x_{31}) \;+\; 25(x_{12}+x_{22}+x_{32}) \\
&\qquad -\; 17.25(x_{11}+x_{12}) \;-\; 15.75(x_{21}+x_{22}) \;-\; 17.75(x_{31}+x_{32}).
\end{aligned}

### Constraints

#### Input availability (each in its own display block)
\begin{aligned}
x_{11} + x_{12} &\le 150
\end{aligned}

\begin{aligned}
x_{21} + x_{22} &\le 350
\end{aligned}

\begin{aligned}
x_{31} + x_{32} &\le 300
\end{aligned}

#### Demand requirements
\begin{aligned}
x_{11} + x_{21} + x_{31} &= 300
\end{aligned}

\begin{aligned}
x_{12} + x_{22} + x_{32} &= 450
\end{aligned}

#### Octane (quality) constraints

Regular (min octane $=90$):
\begin{aligned}
\frac{100x_{11} + 87x_{21} + 110x_{31}}{x_{11}+x_{21}+x_{31}} &\ge 90
\end{aligned}

Equivalent (expanded) — Regular:
\begin{aligned}
100x_{11} + 87x_{21} + 110x_{31} &\ge 90(x_{11}+x_{21}+x_{31})
\end{aligned}

Simplified — Regular:
\begin{aligned}
10x_{11} - 3x_{21} + 20x_{31} &\ge 0
\end{aligned}

Supreme (min octane $=97$):
\begin{aligned}
\frac{100x_{12} + 87x_{22} + 110x_{32}}{x_{12}+x_{22}+x_{32}} &\ge 97
\end{aligned}

Equivalent (expanded) — Supreme:
\begin{aligned}
100x_{12} + 87x_{22} + 110x_{32} &\ge 97(x_{12}+x_{22}+x_{32})
\end{aligned}

Simplified — Supreme:
\begin{aligned}
3x_{12} - 10x_{22} + 13x_{32} &\ge 0
\end{aligned}

#### Non-negativity
\begin{aligned}
x_{11},x_{21},x_{31},x_{12},x_{22},x_{32} &\ge 0
\end{aligned}


# Problem1


In [2]:
prob = LpProblem("Riverside_Oil_Blending", LpMaximize)

Decision Variables

In [3]:
# Variables represent barrels (in 1000) of each input used for each gasoline type
# For Regular gasoline
x11 = LpVariable("Input1_Regular", lowBound=0)  
x21 = LpVariable("Input2_Regular", lowBound=0)  
x31 = LpVariable("Input3_Regular", lowBound=0)  

# For Supreme gasoline
x12 = LpVariable("Input1_Supreme", lowBound=0)  
x22 = LpVariable("Input2_Supreme", lowBound=0)  
x32 = LpVariable("Input3_Supreme", lowBound=0)  

Objective Function

In [4]:
# Profit = Revenue - Costs
# Revenue from Regular = $21 per barrel * total barrels of Regular
# Revenue from Supreme = $25 per barrel * total barrels of Supreme
# Costs = sum of (input cost * barrels used)

prob += (
    21 * (x11 + x21 + x31) +  # Revenue from Regular
    25 * (x12 + x22 + x32) -  # Revenue from Supreme
    17.25 * (x11 + x12) -     # Cost of Input 1
    15.75 * (x21 + x22) -     # Cost of Input 2
    17.75 * (x31 + x32),      # Cost of Input 3
    "Total_Profit"
)

CONSTRAINTS

In [5]:
# Inputs Availability
prob += x11 + x12 <= 150, "Input1_Availability"
prob += x21 + x22 <= 350, "Input2_Availability"
prob += x31 + x32 <= 300, "Input3_Availability"

In [6]:
# Demand Requirements
prob += x11 + x21 + x31 == 300, "Regular_Demand"   
prob += x12 + x22 + x32 == 450, "Supreme_Demand"

In [7]:
# Octane Rating Constraints
# For Regular: weighted average octane >= 90
# (100*x11 + 87*x21 + 110*x31) / (x11 + x21 + x31) >= 90
prob += 100*x11 + 87*x21 + 110*x31 >= 90*(x11 + x21 + x31), "Regular_Octane"
# For Supreme: weighted average octane >= 97
# (100*x12 + 87*x22 + 110*x32) / (x12 + x22 + x32) >= 97
prob += 100*x12 + 87*x22 + 110*x32 >= 97*(x12 + x22 + x32), "Supreme_Octane"

Solve the problem

In [8]:
prob.solve()

1

In [9]:
# DISPLAY RESULTS
print(f"\nStatus: {LpStatus[prob.status]}")


Status: Optimal


In [11]:
print("DECISION VARIABLES (in 1000s of barrels)")
print("\nREGULAR GASOLINE:")
print(f"  Input 1 (Octane 100): {value(x11):,.2f}")
print(f"  Input 2 (Octane 87):  {value(x21):,.2f}")
print(f"  Input 3 (Octane 110): {value(x31):,.2f}")
print(f"  Total Regular:        {value(x11 + x21 + x31):,.2f}")
    
print("\nSUPREME GASOLINE:")
print(f"  Input 1 (Octane 100): {value(x12):,.2f}")
print(f"  Input 2 (Octane 87):  {value(x22):,.2f}")
print(f"  Input 3 (Octane 110): {value(x32):,.2f}")
print(f"  Total Supreme:        {value(x12 + x22 + x32):,.2f}")

DECISION VARIABLES (in 1000s of barrels)

REGULAR GASOLINE:
  Input 1 (Octane 100): 0.00
  Input 2 (Octane 87):  160.87
  Input 3 (Octane 110): 139.13
  Total Regular:        300.00

SUPREME GASOLINE:
  Input 1 (Octane 100): 150.00
  Input 2 (Octane 87):  189.13
  Input 3 (Octane 110): 110.87
  Total Supreme:        450.00


In [13]:
total_input1 = value(x11 + x12)
total_input2 = value(x21 + x22)
total_input3 = value(x31 + x32)    
# Calculate components
regular_revenue = 21 * value(x11 + x21 + x31)
supreme_revenue = 25 * value(x12 + x22 + x32)
total_revenue = regular_revenue + supreme_revenue
    
cost_input1 = 17.25 * total_input1
cost_input2 = 15.75 * total_input2
cost_input3 = 17.75 * total_input3
total_cost = cost_input1 + cost_input2 + cost_input3
    

print(f"Total Revenue:         ${total_revenue:,.2f} (thousands)")
print(f"Total Cost:            ${total_cost:,.2f} (thousands)")
print(f"MAXIMUM PROFIT:        ${value(prob.objective):,.2f} (thousands)")
print(f"                       = ${value(prob.objective) * 1000:,.2f} (dollars)")

Total Revenue:         $17,550.00 (thousands)
Total Cost:            $12,537.50 (thousands)
MAXIMUM PROFIT:        $5,012.50 (thousands)
                       = $5,012,500.00 (dollars)


# Mathematical Formulation for Problem 2

### Sets and Indices
- Months: $t \in \{1,2,3,4,5\}$
- Lease durations: $d \in \{1,2,3,4,5\}$  
Valid only if $t + d - 1 \le 5$.

---

### Parameters
- $S_t$: additional warehouse space needed in month $t$ (thousand sq ft)  
  - $S_1 = 25,\; S_2 = 10,\; S_3 = 20,\; S_4 = 10,\; S_5 = 5$
- $C_d$: cost per 1000 sq ft of a $d$-month lease  
  - $C_1 = 300,\; C_2 = 525,\; C_3 = 775,\; C_4 = 850,\; C_5 = 975$

---

### Decision Variables
- $x_{td}$ = thousands of sq ft leased at the beginning of month $t$ for a duration of $d$ months.

Valid only if $t + d - 1 \le 5$.

---

### Objective Function

\begin{aligned}
\text{Minimize:}\quad
Z &= \sum_{t=1}^{5} \sum_{d=1}^{5} C_d \, x_{td}
\end{aligned}

Expanded:

\begin{aligned}
Z &= 300(x_{11}+x_{21}+x_{31}+x_{41}+x_{51}) \\
&\quad + 525(x_{12}+x_{22}+x_{32}+x_{42}) \\
&\quad + 775(x_{13}+x_{23}+x_{33}) \\
&\quad + 850(x_{14}+x_{24}) \\
&\quad + 975(x_{15})
\end{aligned}

---

### Constraints

#### Month 1 requirement

\begin{aligned}
x_{11} + x_{12} + x_{13} + x_{14} + x_{15} &\ge 25
\end{aligned}

#### Month 2 requirement

\begin{aligned}
x_{12} + x_{13} + x_{14} + x_{15}
+ x_{21} + x_{22} + x_{23} + x_{24} &\ge 10
\end{aligned}

#### Month 3 requirement

\begin{aligned}
x_{13} + x_{14} + x_{15}
+ x_{22} + x_{23} + x_{24}
+ x_{31} + x_{32} + x_{33} &\ge 20
\end{aligned}

#### Month 4 requirement

\begin{aligned}
x_{14} + x_{15}
+ x_{23} + x_{24}
+ x_{32} + x_{33}
+ x_{41} + x_{42} &\ge 10
\end{aligned}

#### Month 5 requirement

\begin{aligned}
x_{15} + x_{24} + x_{33} + x_{42} + x_{51} &\ge 5
\end{aligned}

#### General form (for all $m$)

\begin{aligned}
\sum_{\substack{t,d: \\ t \le m \le t+d-1}} x_{td} &\ge S_m
\end{aligned}

---

### Non-negativity

\begin{aligned}
x_{td} &\ge 0 \quad \text{for all valid } (t,d)
\end{aligned}

---

### Valid Decision Variable Matrix

| Start Month (t) | d=1 | d=2 | d=3 | d=4 | d=5 |
|-----------------|-----|-----|-----|-----|-----|
| 1 | $x_{11}$ | $x_{12}$ | $x_{13}$ | $x_{14}$ | $x_{15}$ |
| 2 | $x_{21}$ | $x_{22}$ | $x_{23}$ | $x_{24}$ | — |
| 3 | $x_{31}$ | $x_{32}$ | $x_{33}$ | — | — |
| 4 | $x_{41}$ | $x_{42}$ | — | — | — |
| 5 | $x_{51}$ | — | — | — | — |

(Empty cells = invalid combinations where $t+d-1 > 5$.)


# Problem 2

In [14]:
# Additional space needed per month (in 1000 sq ft)
space_needed = {1: 25, 2: 10, 3: 20, 4: 10, 5: 5}

# Cost per 1000 sq ft for different lease lengths
lease_costs = {1: 300, 2: 525, 3: 775, 4: 850, 5: 975}

Create the model

In [15]:
prob = LpProblem("Pelletier_Warehouse_Leasing", LpMinimize)

DECISION VARIABLES

In [16]:
x = {}
for start_month in range(1, 6):  # Months 1-5
    for lease_length in range(1, 6):  # Lease lengths 1-5
        # Can only lease if lease doesn't extend beyond month 5
        if start_month + lease_length - 1 <= 5:
            x[start_month, lease_length] = LpVariable(
                f"Lease_Month{start_month}_For{lease_length}Months", 
                lowBound=0
            )

 OBJECTIVE FUNCTION

In [17]:
# Minimize total leasing cost
prob += lpSum([
    lease_costs[lease_length] * x[start_month, lease_length]
    for (start_month, lease_length) in x.keys()
]), "Total_Leasing_Cost"

CONSTRAINTS

In [18]:
# For each month, total leased space must meet the requirement
# Space available in month m = sum of all leases that cover month m

for month in range(1, 6):
    # Find all leases that cover this month
    covering_leases = []
    
    for (start_month, lease_length) in x.keys():
        end_month = start_month + lease_length - 1
        # Check if this lease covers the current month
        if start_month <= month <= end_month:
            covering_leases.append(x[start_month, lease_length])
                
    # The sum of covering leases must meet the requirement for this month
    prob += lpSum(covering_leases) >= space_needed[month], f"Space_Requirement_Month{month}"

Solve the problem

In [20]:
prob.solve()
print(f"\nStatus: {LpStatus[prob.status]}")


Status: Optimal


In [24]:
print("OPTIMAL LEASING DECISIONS")
print(f"{'Starting Month':<15} {'Lease Length':<15} {'Space (1000 sq ft)':<20} {'Cost ($)':<15}")
total_cost = 0
active_leases = []

for (start_month, lease_length) in sorted(x.keys()):
    amount = value(x[start_month, lease_length])
    if amount > 0.01:  
        cost = lease_costs[lease_length] * amount
        total_cost += cost
        active_leases.append((start_month, lease_length, amount, cost))
        print(f"{start_month:<15} {lease_length:<15} {amount:<20.2f} ${cost:<14,.2f}")

print("-" * 80)
print(f"{'TOTAL COST':<50} ${total_cost:,.2f}")


OPTIMAL LEASING DECISIONS
Starting Month  Lease Length    Space (1000 sq ft)   Cost ($)       
1               1               15.00                $4,500.00      
1               4               5.00                 $4,250.00      
1               5               5.00                 $4,875.00      
3               1               10.00                $3,000.00      
--------------------------------------------------------------------------------
TOTAL COST                                         $16,625.00
